# Milestone 2 — ML Models, Tuning, Interpretability and Segmentation

This is a **synthetic demonstration submission** for Milestone 2. It demonstrates a repeatable comparison and clustering workflow. Use the official dataset and justify choices in your final submission.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.cluster import KMeans

df = pd.DataFrame({"income":[42,51,28,75,33,61,25,82,47,39,56,31,66,36,58,29],"loan_amount":[12,15,18,10,16,9,20,11,14,17,13,19,8,16,12,18],"credit_history":[1,1,0,1,0,1,0,1,1,0,1,0,1,0,1,0],"default":[0,0,1,0,1,0,1,0,0,1,0,1,0,1,0,1]})
X, y = df.drop(columns="default"), df["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
pre = ColumnTransformer([("num", Pipeline([("impute", SimpleImputer(strategy="median")),("scale", StandardScaler())]), X.columns.tolist())])
models = {"logistic": LogisticRegression(max_iter=1000, class_weight="balanced"), "random_forest": RandomForestClassifier(random_state=42, class_weight="balanced")}
for name, clf in models.items():
    pipe = Pipeline([("prep", pre), ("model", clf)])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:,1]
    print(name, "ROC-AUC:", round(roc_auc_score(y_test, p), 3), "F1:", round(f1_score(y_test, pipe.predict(X_test)), 3))

search = GridSearchCV(Pipeline([("prep", pre), ("model", RandomForestClassifier(random_state=42, class_weight="balanced"))]), {"model__n_estimators":[50,100], "model__max_depth":[None,3]}, cv=StratifiedKFold(n_splits=2, shuffle=True, random_state=42), scoring="roc_auc")
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_, 3))

# Segmentation is unsupervised; target is excluded from clustering.
cluster_features = StandardScaler().fit_transform(df[["income", "loan_amount", "credit_history"]])
clusters = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(cluster_features)
segmented = df.assign(segment=clusters)
print(segmented.groupby("segment")[["income", "loan_amount", "credit_history"]].mean())
print("Toy data only: validate cluster stability and business meaning on the full dataset.")
